# GastroNet — NB5: `hybrid_crossattn` real multi-seed training

**This is the proposed model.** `hybrid_concat` (NB3) is the ablation
baseline; this notebook trains the actual contribution — unidirectional
cross-attention fusion (CNN spatial tokens as queries, ViT-Small tokens as
keys/values) — and is the comparison that matters most (handoff doc,
Section 3): does cross-attention fusion add anything *over naive
concatenation*, not just "did we beat 98.25%."

**Prerequisites**:
- NB0 already run on this account (`dataset_split_v2.json` +
  `checkpoint_utils.py` present on Drive).
- NB4 already run and passed all shape/gradient sanity checks — this
  notebook reuses that exact architecture verbatim, now with real data and
  real training.
- Ideally `cnn_only_v2`, `vit_only_v2`, `hybrid_concat_v2` already have
  results, so the closing comparison cell here is meaningful immediately.

`MODEL_FAMILY = "hybrid_crossattn_v2"`. Trains 3 seeds (42, 123, 7) with
early stopping — single-seed results are not trusted at these margins
(handoff Section 1/3/9).

**On overfitting** (this model has two pretrained backbones + a fusion
block, more capacity than any prior notebook, over only ~3200 training
images): this notebook adds three defenses *on top of* the already-locked
infra rules (early stopping `PATIENCE=6`, weight decay, multi-seed), none of
which touch dataset/split/checkpoint rules:
1. **Differential learning rates** — backbones fine-tune at a much lower LR
   than the newly-initialized fusion head, so pretrained features aren't
   blown away in a couple of noisy steps.
2. **Short backbone freeze warm-up** — the fusion head trains alone for the
   first couple of epochs so it starts from a sane state before any
   backbone gradients flow.
3. **Label smoothing + gradient clipping** — standard, cheap regularizers
   for a small-dataset transformer-containing model.

These are engineering choices to reduce overfitting risk, not methodology
changes made to chase a target number (handoff Section 3/13) — if
`hybrid_crossattn_v2` still doesn't beat `hybrid_concat_v2`, that's reported
as-is.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os, sys, json

# These must be IDENTICAL to what you set in NB0/NB1/NB2/NB3 on this account.
ACCOUNT_TAG      = "acct_A"
EXPERIMENTS_ROOT = "/content/drive/MyDrive/gastronet_experiments"
NOTEBOOK_NAME    = "NB5_hybrid_crossattn_training"
RAW_DATASET_DIR  = "/content/drive/MyDrive/gastronet_raw_dataset"

# This notebook's own identity within the experiment structure.
MODEL_FAMILY = "hybrid_crossattn_v2"
SEEDS_TO_RUN = [42, 123, 7]

CLASS_NAMES  = ["Diverticulosis", "Neoplasm", "Peritonitis", "Ureters"]
IMG_SIZE_CNN = 448   # EfficientNet-B4 branch native size (Decision B)
IMG_SIZE_VIT = 224   # ViT-Small branch native size (Decision B)

BATCH_SIZE   = 12    # matches NB3's hybrid batch size -- two backbones + two
                      # image tensors/sample use more memory than NB1/NB2.
                      # Drop to 8 if you hit OOM (see handoff Section 12.3).
NUM_EPOCHS   = 30     # ceiling -- early stopping + resume mean this is rarely fully used
PATIENCE     = 6      # epochs without val_acc improvement before stopping early

# Differential learning rates (anti-overfitting choice #1, see intro markdown).
LR_BACKBONE  = 1e-5   # EfficientNet-B4 + ViT-Small pretrained weights
LR_HEAD      = 1e-4   # positional embedding, projections, cross-attn blocks, classifier
WEIGHT_DECAY = 1e-4

WARMUP_EPOCHS   = 2    # backbones frozen for these many epochs (anti-overfitting choice #2)
LABEL_SMOOTHING = 0.1  # anti-overfitting choice #3a
GRAD_CLIP_NORM  = 1.0  # anti-overfitting choice #3b -- also stabilizes attention training

# Points at the CORRECTED split -- do not change back to dataset_split.json (v1).
SPLIT_JSON_PATH    = os.path.join(EXPERIMENTS_ROOT, "dataset_split_v2.json")
MANIFEST_JSON_PATH = os.path.join(EXPERIMENTS_ROOT, "experiments_manifest.json")

assert os.path.exists(SPLIT_JSON_PATH), (
    "dataset_split_v2.json not found. Run NB0's duplicate-check/v2-split cells "
    "on this account first, or copy dataset_split_v2.json + checkpoint_utils.py "
    "in from the account that did."
)
assert os.path.exists(os.path.join(EXPERIMENTS_ROOT, "checkpoint_utils.py")), (
    "checkpoint_utils.py not found in EXPERIMENTS_ROOT. Same fix as above."
)

sys.path.insert(0, EXPERIMENTS_ROOT)
import checkpoint_utils as cku
print("checkpoint_utils imported OK from:", EXPERIMENTS_ROOT)


checkpoint_utils imported OK from: /content/drive/MyDrive/gastronet_experiments


### Local dataset copy
Same rationale as NB1/NB2/NB3: reading thousands of individual files off a
mounted Drive during training triggers Drive API rate-limiting partway
through an epoch. Copy once per session, per-class, per-file, with progress
prints every 200 files. Session-temporary by design.


In [3]:
import shutil

LOCAL_DATASET_DIR = "/content/gastro_local_copy"

if not os.path.exists(LOCAL_DATASET_DIR):
    os.makedirs(LOCAL_DATASET_DIR)
    for cls in CLASS_NAMES:
        src_dir = os.path.join(RAW_DATASET_DIR, cls)
        dst_dir = os.path.join(LOCAL_DATASET_DIR, cls)
        os.makedirs(dst_dir, exist_ok=True)
        files = os.listdir(src_dir)
        print(f"Copying class '{cls}': {len(files)} files")
        for i, fname in enumerate(files):
            shutil.copy2(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))
            if (i + 1) % 200 == 0:
                print(f"  [{cls}] copied {i+1}/{len(files)}")
    print("Local copy complete.")
else:
    print("Local copy already exists this session, skipping copy.")


Copying class 'Diverticulosis': 1000 files
  [Diverticulosis] copied 200/1000
  [Diverticulosis] copied 400/1000
  [Diverticulosis] copied 600/1000
  [Diverticulosis] copied 800/1000
  [Diverticulosis] copied 1000/1000
Copying class 'Neoplasm': 1000 files
  [Neoplasm] copied 200/1000
  [Neoplasm] copied 400/1000
  [Neoplasm] copied 600/1000
  [Neoplasm] copied 800/1000
  [Neoplasm] copied 1000/1000
Copying class 'Peritonitis': 1000 files
  [Peritonitis] copied 200/1000
  [Peritonitis] copied 400/1000
  [Peritonitis] copied 600/1000
  [Peritonitis] copied 800/1000
  [Peritonitis] copied 1000/1000
Copying class 'Ureters': 1000 files
  [Ureters] copied 200/1000
  [Ureters] copied 400/1000
  [Ureters] copied 600/1000
  [Ureters] copied 800/1000
  [Ureters] copied 1000/1000
Local copy complete.


In [4]:
with open(SPLIT_JSON_PATH) as f:
    split_payload = json.load(f)

SPLIT_HASH = split_payload["split_hash"]
split = split_payload["split"]

assert split_payload["class_names"] == CLASS_NAMES, "Class name mismatch with locked split!"

print("Loaded split_hash (v2):", SPLIT_HASH)
if "derived_from" in split_payload:
    print("Derived from v1 split_hash:", split_payload["derived_from"])
    print("Fix note:", split_payload.get("fix_note", ""))
for k in ["train", "val", "test"]:
    print(f"  {k}: {len(split[k])} images")

# Remap paths to the local copy for actual file reads during training.
# dataset_split_v2.json itself is untouched -- SPLIT_HASH above was computed
# from the canonical Drive paths it stores.
def remap_to_local(entries):
    return [(p.replace(RAW_DATASET_DIR, LOCAL_DATASET_DIR), cls) for p, cls in entries]

split = {k: remap_to_local(v) for k, v in split.items()}
print("Paths remapped to local copy, e.g.:", split["train"][0][0])


Loaded split_hash (v2): d6e80caa29bff18856ae93ea635b4650
Derived from v1 split_hash: 5ba9dc1cc68a1202cd0bb0446e457ab0
Fix note: Removed 2 duplicate file(s) that appeared in multiple splits in v1; duplicates kept in test, removed from train/val.
  train: 3198 images
  val: 400 images
  test: 400 images
Paths remapped to local copy, e.g.: /content/gastro_local_copy/Diverticulosis/08648b7f-37b9-406e-b1c3-03683c3222af.jpg


In [5]:
!pip install timm --break-system-packages -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


## `GastroDualDataset` (unchanged from NB3, reused here per Decision B)

Returns `(cnn_tensor_448, vit_tensor_224, label)`. One shared set of
augmentation decisions (flip, rotation, brightness/contrast) is applied
identically to both resized copies of a sample, matching NB1/NB2's
augmentation strength (`RandomRotation(10)`, `ColorJitter(0.1, 0.1)`) so the
fusion models see comparable augmentation intensity to the standalone
baselines -- not a stronger or weaker regime that would confound the
comparison.


In [6]:
from PIL import Image
import random
import torchvision.transforms.functional as TF

class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class GastroDualDataset(torch.utils.data.Dataset):
    """
    Dual native-resolution dataset (Decision B):
      cnn_tensor: 3x448x448 (EfficientNet-B4 branch)
      vit_tensor: 3x224x224 (ViT-Small branch)
    One shared augmentation decision per sample, applied to both resized
    copies -- so both branches see consistent augmented views of the same
    image, not two independently randomized ones.
    """

    def __init__(self, entries, train):
        self.entries = entries
        self.train = train

    def __len__(self):
        return len(self.entries)

    def _augment_params(self):
        return {
            "hflip": self.train and random.random() < 0.5,
            "angle": random.uniform(-10, 10) if self.train else 0.0,
            "brightness": random.uniform(0.9, 1.1) if self.train else 1.0,
            "contrast": random.uniform(0.9, 1.1) if self.train else 1.0,
        }

    def _apply(self, img, size, params):
        img = img.resize((size, size), Image.BILINEAR)
        if params["hflip"]:
            img = TF.hflip(img)
        if params["angle"] != 0.0:
            img = TF.rotate(img, params["angle"])
        if self.train:
            img = TF.adjust_brightness(img, params["brightness"])
            img = TF.adjust_contrast(img, params["contrast"])
        tensor = TF.to_tensor(img)
        tensor = TF.normalize(tensor, IMAGENET_MEAN, IMAGENET_STD)
        return tensor

    def __getitem__(self, idx):
        path, cls = self.entries[idx]
        img = Image.open(path).convert("RGB")
        params = self._augment_params()
        cnn_tensor = self._apply(img, IMG_SIZE_CNN, params)
        vit_tensor = self._apply(img, IMG_SIZE_VIT, params)
        label = class_to_idx[cls]
        return cnn_tensor, vit_tensor, label


train_ds = GastroDualDataset(split["train"], train=True)
val_ds   = GastroDualDataset(split["val"],   train=False)
test_ds  = GastroDualDataset(split["test"],  train=False)

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)
test_loader = torch.utils.data.DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)} "
      f"batches/epoch={len(train_loader)}")


train=3198 val=400 test=400 batches/epoch=267


## Model definition (verbatim from NB4, verified there via shape + gradient checks)

Unidirectional cross-attention: CNN spatial tokens (queries, + learned 2D
positional embedding) attend to ViT-Small patch tokens (keys/values,
content-only, no positional projection). `d_model=256`, ViT's 384 projected
down. `num_layers=2`. Dropout raised to `0.3` here (from NB4's sanity-check
default of `0.1`) to match NB3's fusion-head dropout and because this
notebook trains on real data where overfitting is an active concern, unlike
NB4's dummy-tensor check.


In [7]:
class CNNSpatialEncoder(nn.Module):
    """EfficientNet-B4 backbone, returns the pre-pool spatial feature map."""

    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.EfficientNet_B4_Weights.DEFAULT if pretrained else None
        backbone = torchvision.models.efficientnet_b4(weights=weights)
        self.features = backbone.features  # (B, 1792, H, W) at 448 input
        self.out_channels = 1792

    def forward(self, x):
        return self.features(x)


class ViTTokenEncoder(nn.Module):
    """ViT-Small (timm) backbone, returns patch tokens with CLS dropped."""

    def __init__(self, pretrained=True):
        super().__init__()
        self.vit = timm.create_model(
            "vit_small_patch16_224", pretrained=pretrained, num_classes=0
        )
        self.embed_dim = self.vit.embed_dim  # 384
        self.num_prefix_tokens = getattr(self.vit, "num_prefix_tokens", 1)

    def forward(self, x):
        tokens = self.vit.forward_features(x)
        return tokens[:, self.num_prefix_tokens:, :]  # (B, N_vit, 384)


class Learned2DPositionalEmbedding(nn.Module):
    """Separable row/column learned positional embedding for a HxW grid."""

    def __init__(self, grid_h, grid_w, dim):
        super().__init__()
        self.grid_h = grid_h
        self.grid_w = grid_w
        self.row_embed = nn.Parameter(torch.zeros(grid_h, dim))
        self.col_embed = nn.Parameter(torch.zeros(grid_w, dim))
        nn.init.trunc_normal_(self.row_embed, std=0.02)
        nn.init.trunc_normal_(self.col_embed, std=0.02)

    def forward(self):
        pos = self.row_embed[:, None, :] + self.col_embed[None, :, :]
        return pos.reshape(self.grid_h * self.grid_w, -1)


class CrossAttentionBlock(nn.Module):
    """Unidirectional: query = CNN tokens, key/value = ViT tokens."""

    def __init__(self, d_model=256, num_heads=4, ff_mult=4, dropout=0.3):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * ff_mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * ff_mult, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, cnn_tokens, vit_tokens):
        attn_out, attn_weights = self.attn(
            query=cnn_tokens, key=vit_tokens, value=vit_tokens, need_weights=True
        )
        x = self.norm1(cnn_tokens + self.dropout(attn_out))
        x = self.norm2(x + self.ff(x))
        return x, attn_weights


class HybridCrossAttnModel(nn.Module):
    def __init__(self, num_classes=4, d_model=256, num_heads=4, num_layers=2,
                 cnn_grid=None, dropout=0.3, pretrained_backbones=True):
        super().__init__()
        self.cnn_encoder = CNNSpatialEncoder(pretrained=pretrained_backbones)
        self.vit_encoder = ViTTokenEncoder(pretrained=pretrained_backbones)

        cnn_channels = self.cnn_encoder.out_channels
        vit_dim = self.vit_encoder.embed_dim
        grid_h, grid_w = cnn_grid

        self.cnn_proj = nn.Linear(cnn_channels, d_model)
        self.pos_embed = Learned2DPositionalEmbedding(grid_h, grid_w, d_model)
        self.vit_proj = nn.Linear(vit_dim, d_model)

        self.blocks = nn.ModuleList([
            CrossAttentionBlock(d_model=d_model, num_heads=num_heads, dropout=dropout)
            for _ in range(num_layers)
        ])

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )
        self.grid_h, self.grid_w = grid_h, grid_w

    def forward(self, cnn_input, vit_input, return_attn=False):
        cnn_feat_map = self.cnn_encoder(cnn_input)
        B, C, H, W = cnn_feat_map.shape
        cnn_tokens = cnn_feat_map.flatten(2).transpose(1, 2)
        cnn_tokens = self.cnn_proj(cnn_tokens)
        cnn_tokens = cnn_tokens + self.pos_embed()[None, :, :]

        vit_tokens = self.vit_encoder(vit_input)
        vit_tokens = self.vit_proj(vit_tokens)

        attn_maps = []
        x = cnn_tokens
        for block in self.blocks:
            x, attn_weights = block(x, vit_tokens)
            attn_maps.append(attn_weights)

        pooled = x.mean(dim=1)
        logits = self.classifier(pooled)
        return (logits, attn_maps) if return_attn else logits

    def backbone_parameters(self):
        return list(self.cnn_encoder.parameters()) + list(self.vit_encoder.parameters())

    def head_parameters(self):
        backbone_ids = {id(p) for p in self.backbone_parameters()}
        return [p for p in self.parameters() if id(p) not in backbone_ids]

    def set_backbone_trainable(self, trainable):
        for p in self.backbone_parameters():
            p.requires_grad = trainable


# Probe the CNN's spatial grid size once, exactly as verified in NB4.
_probe_encoder = CNNSpatialEncoder(pretrained=False).to(device).eval()
with torch.no_grad():
    _probe_out = _probe_encoder(torch.randn(2, 3, IMG_SIZE_CNN, IMG_SIZE_CNN, device=device))
CNN_GRID_H, CNN_GRID_W = _probe_out.shape[-2], _probe_out.shape[-1]
print(f"CNN spatial grid: {CNN_GRID_H}x{CNN_GRID_W} = {CNN_GRID_H * CNN_GRID_W} tokens")
del _probe_encoder, _probe_out

def build_crossattn_model():
    return HybridCrossAttnModel(
        num_classes=len(CLASS_NAMES),
        d_model=256,
        num_heads=4,
        num_layers=2,
        cnn_grid=(CNN_GRID_H, CNN_GRID_W),
        dropout=0.3,
        pretrained_backbones=True,
    )

print("Model: hybrid_crossattn, "
      f"{sum(p.numel() for p in build_crossattn_model().parameters()):,} params")


CNN spatial grid: 14x14 = 196 tokens
Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 144MB/s]


model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Model: hybrid_crossattn, 41,359,564 params


## `run_epoch` — dual-input version

Same per-batch progress printing / per-chunk timing pattern as NB1-NB3 (every
~20 batches, chunk time + total elapsed, so a Drive-copy stall or GPU
slowdown is visible within a minute, not after 10+ minutes of silence).
Adapted only to unpack `(cnn_imgs, vit_imgs, labels)` batches and to apply
gradient clipping before the optimizer step.


In [8]:
import time

def run_epoch(model, optimizer, scaler, criterion, loader, train_mode, print_every=20):
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    start_time = time.time()
    last_print_time = start_time

    with torch.set_grad_enabled(train_mode):
        for batch_idx, (cnn_imgs, vit_imgs, labels) in enumerate(loader):
            cnn_imgs = cnn_imgs.to(device, non_blocking=True)
            vit_imgs = vit_imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if train_mode:
                optimizer.zero_grad()

            with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
                outputs = model(cnn_imgs, vit_imgs)
                loss = criterion(outputs, labels)

            if train_mode:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], GRAD_CLIP_NORM
                )
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * cnn_imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += cnn_imgs.size(0)

            if (batch_idx + 1) % print_every == 0:
                now = time.time()
                chunk_time = now - last_print_time
                last_print_time = now
                running_acc = correct / total
                mode_str = "train" if train_mode else "val"
                print(f"    [{mode_str}] batch {batch_idx+1}/{len(loader)} "
                      f"| running_acc={running_acc:.4f} "
                      f"| this_chunk={chunk_time:.1f}s | total_elapsed={now - start_time:.1f}s")

    return total_loss / total, correct / total


## Multi-seed training loop

Same shape as NB1/NB2/NB3: `resume_or_start` -> manifest `started`/`resumed`
-> per-epoch train/val -> `save_latest` every epoch -> `save_best` +
`save_config` on val-accuracy improvement -> `PATIENCE=6` early stopping ->
`KeyboardInterrupt` handling that logs `interrupted` and breaks cleanly ->
test-set eval with `assert_split_hash_matches` first -> `save_results`
(including raw `predictions`/`labels`) -> manifest `completed` log ->
`finalize_experiment` immediately. Skips any seed whose `results.json`
already exists, so this cell is safe to re-run after a Colab disconnect.

Differences from NB1-NB3, all noted in the intro markdown:
- Optimizer uses two param groups (`backbone_parameters()` at `LR_BACKBONE`,
  `head_parameters()` at `LR_HEAD`) instead of one flat LR.
- `model.set_backbone_trainable(epoch >= WARMUP_EPOCHS)` is called at the
  top of every epoch (not just once) so the freeze schedule is correctly
  re-applied after a resume, regardless of which epoch training resumes at.
- `criterion` uses `label_smoothing=LABEL_SMOOTHING`.
- Gradient clipping happens inside `run_epoch` (see above).


In [9]:
all_seed_results = {}

for SEED in SEEDS_TO_RUN:
    seed_exp_dir = cku.get_experiment_dir(EXPERIMENTS_ROOT, MODEL_FAMILY, SEED)

    if os.path.exists(os.path.join(seed_exp_dir, "results.json")):
        print(f"Seed {SEED} already has results.json, skipping.")
        with open(os.path.join(seed_exp_dir, "results.json")) as f:
            all_seed_results[SEED] = json.load(f)
        continue

    print(f"\n{'='*60}\nStarting SEED={SEED}\n{'='*60}")

    torch.manual_seed(SEED)
    model = build_crossattn_model().to(device)

    optimizer = torch.optim.AdamW(
        [
            {"params": model.backbone_parameters(), "lr": LR_BACKBONE},
            {"params": model.head_parameters(), "lr": LR_HEAD},
        ],
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    start_epoch, best_val_acc, history = cku.resume_or_start(
        seed_exp_dir, model, optimizer, scheduler, scaler, map_location=device
    )
    cku.log_run_event(MANIFEST_JSON_PATH, MODEL_FAMILY, SEED, ACCOUNT_TAG, NOTEBOOK_NAME,
                       status="resumed" if start_epoch > 0 else "started",
                       best_val_acc=best_val_acc, drive_path=seed_exp_dir)

    epochs_without_improvement = 0

    try:
        for epoch in range(start_epoch, NUM_EPOCHS):
            backbone_trainable = epoch >= WARMUP_EPOCHS
            model.set_backbone_trainable(backbone_trainable)
            if epoch == 0 or epoch == WARMUP_EPOCHS:
                state = "trainable" if backbone_trainable else "FROZEN (warm-up)"
                print(f"  [seed {SEED}] backbones {state} as of epoch {epoch+1}")

            train_loss, train_acc = run_epoch(model, optimizer, scaler, criterion, train_loader, train_mode=True)
            val_loss, val_acc = run_epoch(model, optimizer, scaler, criterion, val_loader, train_mode=False)
            scheduler.step()

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            print(f"[seed {SEED}] Epoch {epoch+1}/{NUM_EPOCHS} | "
                  f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
                  f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

            gap = train_acc - val_acc
            if gap > 0.10:
                print(f"  -> NOTE: train/val accuracy gap = {gap:.3f} "
                      f"(train pinning high while val lags is the overfitting "
                      f"pattern flagged in NB1 -- PATIENCE={PATIENCE} will stop "
                      f"this once val stops improving)")

            cku.save_latest(seed_exp_dir, epoch, model, optimizer, scheduler, scaler, best_val_acc, history)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                epochs_without_improvement = 0
                config = {
                    "model_family": MODEL_FAMILY, "seed": SEED, "split_hash": SPLIT_HASH,
                    "account_tag": ACCOUNT_TAG, "notebook_name": NOTEBOOK_NAME,
                    "cnn_img_size": IMG_SIZE_CNN, "vit_img_size": IMG_SIZE_VIT,
                    "batch_size": BATCH_SIZE, "lr_backbone": LR_BACKBONE, "lr_head": LR_HEAD,
                    "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
                    "label_smoothing": LABEL_SMOOTHING, "grad_clip_norm": GRAD_CLIP_NORM,
                    "d_model": 256, "num_heads": 4, "num_layers": 2,
                    "cnn_grid": [CNN_GRID_H, CNN_GRID_W], "class_names": CLASS_NAMES,
                }
                cku.save_best(seed_exp_dir, model, epoch, best_val_acc, config)
                cku.save_config(seed_exp_dir, config)
                print(f"  -> new best_val_acc={best_val_acc:.4f}, saved best.pt")
            else:
                epochs_without_improvement += 1
                print(f"  -> no improvement for {epochs_without_improvement}/{PATIENCE} epochs")
                if epochs_without_improvement >= PATIENCE:
                    print(f"Stopping early for seed {SEED}: no improvement for {PATIENCE} epochs.")
                    cku.save_history(seed_exp_dir, history)
                    break

            cku.save_history(seed_exp_dir, history)

    except KeyboardInterrupt:
        cku.log_run_event(MANIFEST_JSON_PATH, MODEL_FAMILY, SEED, ACCOUNT_TAG, NOTEBOOK_NAME,
                           status="interrupted", best_val_acc=best_val_acc, drive_path=seed_exp_dir,
                           note="Manually interrupted -- latest.pt has current state, safe to resume.")
        print(f"Interrupted during seed {SEED}. Re-run this cell to resume from the last completed epoch.")
        break

    # Test-set evaluation for this seed
    best_ckpt = cku.load_best(seed_exp_dir, map_location=device)
    cku.assert_split_hash_matches(best_ckpt["config"], SPLIT_HASH)
    eval_model = build_crossattn_model().to(device)
    eval_model.load_state_dict(best_ckpt["model_state_dict"])
    eval_model.eval()

    correct, total, preds_all, labels_all = 0, 0, [], []
    with torch.no_grad():
        for cnn_imgs, vit_imgs, labels in test_loader:
            cnn_imgs, vit_imgs, labels = cnn_imgs.to(device), vit_imgs.to(device), labels.to(device)
            preds = eval_model(cnn_imgs, vit_imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += cnn_imgs.size(0)
            preds_all.extend(preds.cpu().tolist())
            labels_all.extend(labels.cpu().tolist())

    test_acc = correct / total
    results = {
        "model_family": MODEL_FAMILY, "seed": SEED, "split_hash": SPLIT_HASH,
        "account_tag": ACCOUNT_TAG, "best_epoch": best_ckpt["epoch"],
        "best_val_acc": best_ckpt["best_val_acc"], "test_accuracy": test_acc,
        "predictions": preds_all, "labels": labels_all, "class_names": CLASS_NAMES,
    }
    cku.save_results(seed_exp_dir, results)
    all_seed_results[SEED] = results

    cku.log_run_event(MANIFEST_JSON_PATH, MODEL_FAMILY, SEED, ACCOUNT_TAG, NOTEBOOK_NAME,
                       status="completed", best_val_acc=best_val_acc, drive_path=seed_exp_dir)

    cku.finalize_experiment(seed_exp_dir)
    print(f"Seed {SEED} done. test_acc={test_acc:.4f}. latest.pt cleaned up.\n")

print("\nSeeds completed this run:")
for s, r in all_seed_results.items():
    print(f"  seed {s}: test_accuracy={r['test_accuracy']:.4f}, best_val_acc={r['best_val_acc']:.4f}")



Starting SEED=42


/tmp/ipykernel_864/3807913519.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


[/content/drive/MyDrive/gastronet_experiments/hybrid_crossattn_v2/seed_42] No latest.pt found -- starting fresh at epoch 0.
  [seed 42] backbones FROZEN (warm-up) as of epoch 1
    [train] batch 20/267 | running_acc=0.5208 | this_chunk=13.5s | total_elapsed=13.5s
    [train] batch 40/267 | running_acc=0.6354 | this_chunk=6.5s | total_elapsed=20.0s
    [train] batch 60/267 | running_acc=0.7125 | this_chunk=7.4s | total_elapsed=27.4s
    [train] batch 80/267 | running_acc=0.7521 | this_chunk=6.1s | total_elapsed=33.5s
    [train] batch 100/267 | running_acc=0.7842 | this_chunk=6.2s | total_elapsed=39.7s
    [train] batch 120/267 | running_acc=0.8076 | this_chunk=5.4s | total_elapsed=45.1s
    [train] batch 140/267 | running_acc=0.8202 | this_chunk=6.9s | total_elapsed=51.9s
    [train] batch 160/267 | running_acc=0.8318 | this_chunk=5.1s | total_elapsed=57.0s
    [train] batch 180/267 | running_acc=0.8375 | this_chunk=6.0s | total_elapsed=63.0s
    [train] batch 200/267 | running_acc=0.8

In [10]:
cku.manifest_summary(MANIFEST_JSON_PATH)


_setup             seed_0      -> completed    acc=None account=acct_A notebook=NB0_setup_and_split_lock at 2026-09-07 10:15:46
cnn_only           seed_7      -> completed    acc=0.985 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-06 12:56:38
cnn_only           seed_42     -> resumed      acc=0.9825 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-06 12:26:57
cnn_only           seed_123    -> completed    acc=0.9825 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-06 12:42:24
cnn_only_v2        seed_7      -> completed    acc=0.9825 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-07 11:45:05
cnn_only_v2        seed_42     -> completed    acc=0.985 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-07 11:01:48
cnn_only_v2        seed_123    -> completed    acc=0.98 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-07 11:18:20
hybrid_crossattn_v2 seed_7      -> completed    acc=0.9775 account=acct_A notebook=NB5_hybrid_crossattn_training a

In [11]:
import numpy as np
import glob

all_results = []
for rf in sorted(glob.glob(os.path.join(EXPERIMENTS_ROOT, MODEL_FAMILY, "seed_*", "results.json"))):
    with open(rf) as f:
        all_results.append(json.load(f))

test_accs = [r["test_accuracy"] for r in all_results]
seeds_found = [r["seed"] for r in all_results]

print(f"{MODEL_FAMILY} -- seeds found: {seeds_found}")
print(f"Test accuracies: {[round(a, 4) for a in test_accs]}")
if len(test_accs) > 1:
    print(f"Mean: {np.mean(test_accs):.4f}  Std: {np.std(test_accs, ddof=1):.4f}")
else:
    print("Only one seed found so far.")


hybrid_crossattn_v2 -- seeds found: [123, 42, 7]
Test accuracies: [0.97, 0.96, 0.975]
Mean: 0.9683  Std: 0.0076


In [ ]:
# FAMILIES_TO_COMPARE = ["cnn_only_v2", "vit_only_v2", "hybrid_concat_v2", "hybrid_crossattn_v2"]
# PAPER_BASELINE = {"name": "Paper: Proposed GastroNetV4", "accuracy": 0.9825}

# print(f"{'model_family':<20} {'seeds':<18} {'mean':<8} {'std':<8}")
# print("-" * 56)
# for fam in FAMILIES_TO_COMPARE:
#     rfs = sorted(glob.glob(os.path.join(EXPERIMENTS_ROOT, fam, "seed_*", "results.json")))
#     if not rfs:
#         print(f"{fam:<20} {'no results yet':<18}")
#         continue
#     accs = []
#     seeds = []
#     for rf in rfs:
#         with open(rf) as f:
#             r = json.load(f)
#         accs.append(r["test_accuracy"])
#         seeds.append(r["seed"])
#     mean = np.mean(accs)
#     std = np.std(accs, ddof=1) if len(accs) > 1 else float("nan")
#     print(f"{fam:<20} {str(seeds):<18} {mean:<8.4f} {std:<8.4f}")

# print("-" * 56)
# print(f"{PAPER_BASELINE['name']:<38} {PAPER_BASELINE['accuracy']:.4f}")
# print()
# print("Reminder (handoff Section 3): the key question is whether "
#       "hybrid_crossattn_v2 clearly beats hybrid_concat_v2, not whether "
#       "either beats the paper's 0.9825. If they land within noise of each "
#       "other, that itself is the honest, reportable finding -- carried into "
#       "NB6's significance testing, not tinkered with here.")


## After this notebook finishes

1. **Check for overfitting directly**: open `history.json` for each seed in
   `gastronet_experiments/hybrid_crossattn_v2/seed_{42,123,7}/` and look at
   the train/val accuracy gap over epochs — the training loop above already
   flags any epoch where that gap exceeds 0.10. `PATIENCE=6` should have
   stopped training before this became severe, but worth a glance,
   especially since this model has the most capacity of any notebook so far.
2. **Check `results.json`** for each seed — `test_accuracy`/`best_val_acc`
   sane, `predictions`/`labels` present for NB6's significance testing.
3. **Mean±std + comparison table**: printed directly above. Compare
   `hybrid_crossattn_v2` against `hybrid_concat_v2` first — that's the
   comparison the project's contribution actually rests on (Section 3).
4. **If Colab disconnects mid-run**: re-run the multi-seed training cell —
   it skips any seed with an existing `results.json` and resumes the
   in-progress seed from `latest.pt` via `resume_or_start`. The backbone
   freeze schedule is recomputed from the current epoch every time, so
   resuming mid-warm-up or mid-fine-tuning is handled correctly either way.
5. **If `hybrid_crossattn_v2` doesn't clearly beat `hybrid_concat_v2`**:
   that's a valid, reportable finding per Section 3
6. **Next: NB6 (final analysis)** — load every model_family's
   `seed_*/results.json` (v2 only), full mean±std comparison table against
   the paper's Table 3, and a paired significance test (e.g. McNemar's)
   between the closest-performing models. Only after NB6 is locked does
   NB7+ (explainability: Grad-CAM, cross-attention maps, t-SNE, calibration)
   start.
